**Lab type:** debug

**Course:** ML201 — Applied Machine Learning

**Lesson:** Handling Imbalanced Data

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    precision_recall_curve, recall_score, precision_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

np.random.seed(42)
n = 5000

# Fraud detection dataset: ~2% positive class
fraud_rate = 0.02
n_fraud = int(n * fraud_rate)
n_legit = n - n_fraud

# Legitimate transactions (label 0)
legit_amount      = np.random.lognormal(mean=4.5, sigma=1.0, size=n_legit)
legit_age         = np.random.randint(30, 3650, size=n_legit)
legit_num_tx      = np.random.poisson(lam=12, size=n_legit)
legit_weekend     = np.random.binomial(1, 0.3, size=n_legit)
legit_merchant    = np.random.randint(0, 10, size=n_legit)

# Fraudulent transactions (label 1) — different distributions
fraud_amount      = np.random.lognormal(mean=5.5, sigma=1.5, size=n_fraud)
fraud_age         = np.random.randint(30, 730, size=n_fraud)
fraud_num_tx      = np.random.poisson(lam=3, size=n_fraud)
fraud_weekend     = np.random.binomial(1, 0.6, size=n_fraud)
fraud_merchant    = np.random.randint(0, 10, size=n_fraud)

X_legit  = np.column_stack([legit_amount, legit_age, legit_num_tx, legit_weekend, legit_merchant])
X_fraud  = np.column_stack([fraud_amount, fraud_age, fraud_num_tx, fraud_weekend, fraud_merchant])

X = np.vstack([X_legit, X_fraud])
y = np.hstack([np.zeros(n_legit), np.ones(n_fraud)]).astype(int)

feature_names = ["transaction_amount", "account_age_days", "num_transactions_30d", "is_weekend", "merchant_category"]
df = pd.DataFrame(X, columns=feature_names)
df["fraud"] = y

# Shuffle
shuffle_idx = np.random.permutation(n)
X = X[shuffle_idx]
y = y[shuffle_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Fraud detection dataset")
print("=" * 40)
print(f"Total samples:  {n}")
print(f"Fraud (label 1): {y.sum()} ({y.mean()*100:.1f}%)")
print(f"Legit (label 0): {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"Train: {len(y_train)}  |  Test: {len(y_test)}")
print(f"Fraud in test: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

## Step 1: Accuracy as the metric on imbalanced data

An analyst trains a logistic regression model on the fraud dataset and celebrates 98% accuracy.

In [ ]:
# Note: uses X_train, X_test, y_train, y_test from Setup

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))  # <- Bug 1: accuracy meaningless on 2% positive class
print(f"Accuracy: {acc:.3f}")
print("Excellent performance!")

**Bug 1 Investigation:** Consider a classifier that predicts "not fraud" for every single transaction, regardless of the features. What accuracy score would it achieve on this test set? Does `accuracy_score` tell you anything useful about whether the model is actually catching fraud? What metric(s) should be used instead for a 2% positive class problem?

*(Write your answer here.)*

In [ ]:
# Fix Bug 1 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What was wrong:** On a 2% positive class, a classifier that predicts "not fraud" for every transaction achieves 98% accuracy — identical to the reported result. Accuracy rewards the model for doing nothing useful. It is dominated by the majority class and cannot distinguish a meaningful fraud detector from a trivial "always predict negative" baseline.

**Why the reported AUC is misleading:** The 98% figure masks the fact that the model may be catching zero fraud cases. Precision, recall, and F1 for the positive class are the right metrics: they measure whether the model actually identifies fraud. ROC AUC or the precision-recall curve are also appropriate because they evaluate the model's discrimination ability across all thresholds, not just at the default 0.5 cutoff.

**Correct approach:** Report `classification_report` (showing per-class precision, recall, F1) and `roc_auc_score`. For 2% positive-class problems, precision-recall AUC is often more informative than ROC AUC.

</details>

## Step 2: SMOTE applied before train/test split

An analyst applies SMOTE to the full dataset to balance the classes, then splits into train and test.

In [ ]:
# Note: uses X, y from Setup

from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)  # <- Bug 2: SMOTE applied to full dataset before split

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42
)

model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X_train2, y_train2)

auc2 = roc_auc_score(y_test2, model2.predict_proba(X_test2)[:, 1])
print(f"AUC: {auc2:.3f}")

**Bug 2 Investigation:** SMOTE generates synthetic minority-class points by interpolating between real minority-class examples and their k-nearest neighbours. When SMOTE is applied to the full dataset before the split, some synthetic training points are generated using the neighbourhood of test-set examples. What does this mean for the reported AUC? Is `X_test2` truly unseen data from the model's perspective?

*(Write your answer here.)*

In [ ]:
# Fix Bug 2 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What was wrong:** SMOTE creates synthetic minority-class points by interpolating between real fraud examples and their k-nearest neighbours in the full dataset. When applied before splitting, the synthetic training points were generated using the neighbourhood of *test-set* fraud examples — meaning the training data is no longer independent of the test data. Some training examples are linear combinations of test-set features.

**Why X_test2 is not truly unseen:** The test set's minority-class examples influenced which synthetic points ended up in the training set. The model has effectively been exposed to information about the test distribution during training, making the reported AUC optimistically biased.

**Correct approach:** Apply SMOTE inside an `ImbPipeline`, then pass the pipeline to `cross_val_score` with `StratifiedKFold`. SMOTE is re-fit inside each fold on that fold's training data only — the validation fold is never used in synthetic point generation.

</details>

## Step 3: Fixed threshold of 0.5 ignoring business cost ratio

After training a model, the analyst uses the default 0.5 probability threshold and reports the model is ready to deploy because "recall is acceptable."

In [ ]:
# Note: uses X_train, X_test, y_train, y_test from Setup
# Train a class-aware model so predictions are non-trivial
model3 = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
model3.fit(X_train, y_train)

probs = model3.predict_proba(X_test)[:, 1]
y_pred = (probs >= 0.5).astype(int)  # <- Bug 3: 0.5 threshold inappropriate for 2% positive class

rec = recall_score(y_test, y_pred)
print(f"Recall: {rec:.3f}")
print("Ready to deploy — recall is acceptable.")

**Bug 3 Investigation:** In fraud detection, a missed fraud case (false negative) typically costs far more than a false alarm (false positive). Given that the positive class is only 2% of transactions, is there any reason to believe 0.5 is a sensible default decision threshold? What business information would you need to choose a threshold rationally, and how would you search for it using the model's predicted probabilities?

*(Write your answer here.)*

In [ ]:
# Fix Bug 3 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What was wrong:** With only 2% positive class, a well-calibrated model assigns low probabilities to most transactions — many true fraud cases may have predicted probabilities of 0.1–0.3. A hard cutoff at 0.5 will miss most of them. The 0.5 threshold has no business justification; it is an arbitrary convention borrowed from balanced datasets.

**How to choose rationally:** Define the cost ratio — how much more expensive is a missed fraud (false negative) than a false alarm (false positive)? In fraud detection, this is typically 10:1 to 50:1. Use the precision-recall curve: sweep all possible thresholds, compute the expected business cost at each (FN_count × cost_fn + FP_count × cost_fp), and select the threshold that minimises total cost. This threshold will typically be much lower than 0.5 for rare-class problems.

</details>

## Corrected Analysis

The cell below applies all three fixes in the correct order. Run it end-to-end to confirm the pipeline is now sound.

In [ ]:
# Note: uses X_train, X_test, y_train, y_test from Setup

# ---- Fix 1: Use class_weight='balanced', report precision/recall/F1 and AUC ----
model_fixed = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
model_fixed.fit(X_train, y_train)

y_pred_fixed_default = model_fixed.predict(X_test)
probs_fixed = model_fixed.predict_proba(X_test)[:, 1]

print("Fix 1 — Correct metrics for imbalanced data:")
print(classification_report(y_test, y_pred_fixed_default, target_names=["legit", "fraud"]))
print(f"ROC AUC: {roc_auc_score(y_test, probs_fixed):.4f}")
print()

# ---- Fix 2: SMOTE inside ImbPipeline, evaluated with StratifiedKFold ----
pipe_smote = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(pipe_smote, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
print("Fix 2 — SMOTE inside Pipeline (5-fold stratified CV on train set):")
print(f"  AUC: {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})")
print()

# ---- Fix 3: Find threshold achieving recall >= 0.80 ----
precisions, recalls, thresholds = precision_recall_curve(y_test, probs_fixed)

# Find the highest threshold where recall >= 0.80
target_recall = 0.80
valid = [(t, p, r) for t, p, r in zip(thresholds, precisions[:-1], recalls[:-1]) if r >= target_recall]

if valid:
    best_threshold, best_precision, best_recall = max(valid, key=lambda x: x[0])
else:
    best_threshold = thresholds[np.argmax(recalls[:-1] >= target_recall)]
    best_precision = precisions[np.argmax(recalls[:-1] >= target_recall)]
    best_recall    = recalls[np.argmax(recalls[:-1] >= target_recall)]

y_pred_tuned = (probs_fixed >= best_threshold).astype(int)

print(f"Fix 3 — Threshold tuned for recall >= {target_recall}:")
print(f"  Selected threshold: {best_threshold:.4f}")
print(f"  Precision at threshold: {best_precision:.4f}")
print(f"  Recall at threshold:    {best_recall:.4f}")
print()
print("Classification report at tuned threshold:")
print(classification_report(y_test, y_pred_tuned, target_names=["legit", "fraud"]))

# Precision-recall curve plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recalls, precisions, color="steelblue", linewidth=2, label="Precision-Recall curve")
ax.axvline(best_recall, color="darkorange", linestyle="--", label=f"Chosen threshold (recall={best_recall:.2f})")
ax.scatter([best_recall], [best_precision], color="darkorange", zorder=5)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — Fraud Detection")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print()
print("All three issues have been resolved.")
print("  1. Correct metrics (precision/recall/F1/AUC) replace misleading accuracy.")
print("  2. SMOTE is applied inside the cross-validation loop — no data leakage.")
print("  3. Threshold chosen from the precision-recall curve, not defaulted to 0.5.")